In [ ]:
from pyspark.sql.functions import *
import dlt

In [ ]:
def apply_gold_schema(df: DataFrame, source: str) -> DataFrame:

    df = df.withColumn("trip_hash_id", sha2(concat(col("id"), lit("_"), lit(source)), 256))

    base_cols = [
        col("id").alias("origin_id"),
        col("trip_hash_id"),
        col("vendorid"),
        col("passenger_count"),
        col("total_amount"),
        col("pickup_datetime"),
        col("dropoff_datetime"),
        col("date_partition"),
        lit(source).alias("cabs_type")
    ]

    return df.select(*base_cols)

In [ ]:
@dlt.table(
    name="fact_taxi_trips",
    comment="Taxi trips from yellow and green cabs",
    partition_cols=["date_partition"]
)
def fact_taxi_trips():
    green_df = dlt.read("silver_green_taxi")
    yellow_df = dlt.read("silver_yellow_taxi")

    green_gold = apply_gold_schema(green_df, "green_taxi")
    yellow_gold = apply_gold_schema(yellow_df, "yellow_taxi")

    return green_gold.unionByName(yellow_gold)